# 云端真实票据评估
适用于 RunPod、AutoDL 等 Linux NVIDIA GPU + Jupyter 环境。先将项目 ZIP 解压到云主机，再从项目目录打开本笔记本。

先跑验证集，分析问题后再跑测试集。当前没有预填模型分数。运行结束后仍需手动结束 GPU 租用。

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
candidates = [Path.cwd(), Path.cwd().parent]
root = next((p for p in candidates if (p / "evaluate.py").exists()), None)
if root is None:
    raise RuntimeError("请先解压完整项目，并从项目内打开 notebook")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("项目目录:", root)


## 安装并检查环境
运行此格会从网络安装项目依赖。

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install",
                "-r", "requirements-local.txt", "-r", "requirements-eval.txt"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("没有检测到 NVIDIA GPU，请确认云实例及 CUDA PyTorch 环境")
print(torch.cuda.get_device_name(0))


## 准备验证集
从官方 CORD v2 下载验证集，在清单中记录不可变的源版本。

In [ ]:
dataset = Path("data/cord/validation/annotations.jsonl")
if not dataset.exists():
    subprocess.run([sys.executable, "scripts/prepare_cord.py", "--split", "validation"], check=True)
print(json.loads(dataset.with_name("manifest.json").read_text()))


## 首轮推理
第一轮使用 20 个验证样本。模型在云 GPU 加载，完整输出保存在新目录中。

In [ ]:
os.environ["MLLM_BACKEND"] = "transformers"
os.environ["MLLM_LOCAL_MODEL"] = "Qwen/Qwen2.5-VL-3B-Instruct"
os.environ["MLLM_LOAD_IN_4BIT"] = "true"
os.environ["MLLM_MIN_PIXELS"] = "200704"
os.environ["MLLM_MAX_PIXELS"] = "802816"
os.environ["MLLM_MAX_NEW_TOKENS"] = "512"
from datetime import datetime, timezone
run_dir = Path("artifacts/runs") / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ") + "-validation")
from evaluate import main
exit_code = main(["--dataset", str(dataset), "--limit", "20", "--output", str(run_dir)])
print("退出码:", exit_code, "（1 表示至少有一次推理失败，请查看报告）")


## 查看错误并保存报告
逐条查看错误原图和输出。模型给出的 confidence 不是经过校准的概率。

In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown((run_dir / "summary.md").read_text()))
report = json.loads((run_dir / "run.json").read_text())
errors = [r for r in report["results"] if r["error"] or r["score"]["correct"] < r["score"]["total"]]
for row in errors[:5]:
    display(Image(filename=str(dataset.parent / row["image"]), width=500))
    print(row["id"], json.dumps(row["score"]["details"], ensure_ascii=False, indent=2))
    print("原始输出:", (row["prediction"] or {}).get("raw_output") or row.get("raw_error_output"))


In [ ]:
import shutil
archive = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
print("请从 Jupyter 文件列表下载:", archive)


## 最终测试集（默认关闭）
先完成验证集分析并冻结配置，再将 RUN_FINAL_TEST 改为 True。不要使用测试集反复选择提示词或参数。

In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    subprocess.run(["bash", "scripts/run_cloud.sh", "test"], check=True)
else:
    print("尚未运行最终测试集。请先完成验证集分析。")


下载报告后，在云平台停止或终止实例并检查存储费用。这个 notebook 不会自动关机。